# Import and Shared functions

In [ ]:
import os, sys
import math
import re
import pandas as pd
import numpy as np
import ephem
from scipy import signal
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from dateutil import parser as dateutil_parser
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import kinematics
import control
import analyse_helpers
importlib.reload(kinematics)
importlib.reload(control)
importlib.reload(analyse_helpers)
from kinematics import calc_parallactic_angle, azaltroll_to_theta, apply_mechanical_corrections, azaltroll_to_q, MountModelParams
from control import theta_to_jacobian, PecMode
from quaternion import Q as Quaternion
from analyse_helpers import resolve_log_files, load_pec, load_kf_pid, load_phd2_guidelog, window_df, load_sync_guiding_residuals

ARCSEC = 3600  # deg -> arcsec, same convention as analyse_kf_pid.ipynb

def r2_score(y, yhat):
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

def r2_quality(r2):
    return (
        "excellent" if r2>0.95 else 
        "good" if r2>0.8 else 
        "moderate" if r2>0.5 else 
        "weak" if r2>0.25 else 
        "poor")

def pec_quality_db(snr_db):
    return (
        "excellent" if snr_db > 30 else
        "good"      if snr_db > 20 else
        "moderate"  if snr_db > 10 else
        "weak"      if snr_db > 5  else
        "poor"
    )


# Load data

In [ ]:
# ── Choose correct log path (last set log_filenames is what is used) ──────────────────────────────────
# Single file:            'alpaca.log'                     (relative to LOG_DIR below)
# All rotated files:      'alpaca.log*'                    (glob -- handles a multi-hour run spanning several files)
# Explicit file list:     ['alpaca.log.2', 'alpaca.log.1', 'alpaca.log']
# A full/relative path (as before) also still works and bypasses LOG_DIR: '../logs/alpaca.log'
LOG_DIR = '../logs/archive'   # base directory log_filenames below are resolved against

log_filenames = 'alpaca.sga_omega_cent.log' # Omega Centuri 1st session
log_filenames = 'alpaca.sga_lagoon.log'     # Lagoon Nebula 2nd session
log_filenames = 'alpaca.sga_pulsepec.log'   # Pulse Guiding with PEC on Beehive cluster
log_filenames = 'alpaca.sga_eagle.log'      # Sync Guiding with PEC on Eagle Nebula
log_filenames = 'alpaca.sga_catspaw.log'      # Sync Guiding with PEC on Eagle Nebula
log_filenames = 'alpaca.pecSlide.acrux2.log'
log_filenames = 'vlogs/alpaca.log.4'       
log_filenames = 'alpaca.h1.h0.log'
log_filenames = 'alpaca.pec_rls_h2.log'
#log_filenames = 'alpaca.pec_ema.log'
#log_filenames = 'alpaca.pec_rls_7hrs.log'
#log_filenames = 'alpaca.pec_rls_6hrs.log'
#log_filenames = 'alpaca.mark_Beta3_08_02.log'
log_filenames = 'alpaca.pec_master.log'
log_filenames = 'alpaca.mark_Beta3_08_02.log'
log_filenames = 'alpaca.pecSin.chicken.log' # PHD2 guiding, somewhat ok
log_filenames = 'alpaca.log'
log_filenames = ['alpaca.soak_pec_Beta4.3_08_30a*.log']  # glob inside a list, like analyse_kf_pid.ipynb
log_filenames = ['alpaca.soak_pec_Beta4.3_08_29m*.log']  # glob inside a list, like analyse_kf_pid.ipynb
log_filenames = ['alpaca.pec_rls_Alpha_06_03_pulseguide_pec.log']
log_filenames = ['alpaca.soak_pec_Beta4.3_08_31*.log']
log_filenames = ['alpaca.soak_Beta4.4_09_01_sg_sglog_Dec10_a*.log']                 # sync guiding,     with logs(sg), Eagle Nebula
log_filenames = ['alpaca.soak_Beta4.4_09_03_sgpec_sgkfposlog_m83_a*.log']           # sync guiding,     with logs(sg), M83 Southern Pinwheel Galaxy      06:00PM - 09:30PM
log_filenames = ['alpaca.soak_Beta4.4_09_03_sgpec_sglog_NGC55_b*.log']              # sync guiding, pec with logs(sg), NGC55 String of Pearls Galaxy    09:45PM - 11:45PM
log_filenames = ['alpaca.soak_Beta4.4_09_03_sgpec_sgkfposlog_ngc2070_c*.log']       # sync guiding, pec with logs(sg,kf,pos), NGC2070 Tarantula Nebula  11:46PM - 02:30AM
log_filenames = ['alpaca.soak_Beta4.4_09_04_sgpec_sgkfposlog_ngc4945_a*.log']       # sync guiding, pec with logs(sg,kf,pos), NGC4945 Tweezers Galaxy   06:30PM - 08:00PM pec_tau_sec=420
log_filenames = ['alpaca.mark_Beta4.3_09_04_a*.log']                                # sync guiding      with logs(sg,kf,pos), 47 Tuc Marks log          2hr 47m session      
log_filenames = ['alpaca.mark_Beta4.4_09_05_a*.log']                                # sync guiding, pec with logs(sg,kf,pos), 47 Tuc Marks log          2hr 30m session  
#log_filenames = ['alpaca.soak_Beta4.4_09_06_sg_sglog_a1.log']                       # sync guiding, pec with logs(sg,pec),  2hr Helix Nebula, then 37m Caldwell 101 session  
#log_filenames = ['alpaca.soak_Beta4.4_09_08_sg_sglog_a1.log']                       # sync guiding,     with logs(sg),  a lot of 518 lags session  
log_filenames = ['alpaca.vyskocil_Beta5.0_09_12_a*.log']                             # pulse guiding, pec with logs(pec)
log_filenames = ['alpaca.jdm_Beta7.1_09_29_pulseguide_a1.log']                      # pulse guiding (PHD2), pec + sync guiding, 3.4 hr on one target from 01:24

# ── Optional time window (local time, as written in the log) -- None = whole log ─────────────────────
# Cut a multi-target night down to one PEC run, e.g. between gotos.
T_FROM = None
T_TO   = None
T_FROM = '2026-09-30 00:50'      # jdm_Beta7.1_09_29: last target -- PEC off 00:51-01:24, on 01:24-04:49

# ── Optional PHD2 guide log for the same session (relative to LOG_DIR, or a full path) ───────────────
# None skips the "Guider Outcome" section; everything else only needs the driver log.
PHD2_GUIDELOG = None
PHD2_GUIDELOG = 'alpaca.jdm_Beta7.1_09_29_pulseguide_a1_PHD2_GuideLog_2026-09-29_185315.txt'


In [ ]:
# ── Read and Parse PECLOG data from log file(s) ───────────────────────────────────────────────────────────
resolved_files = resolve_log_files(log_filenames, log_dir=LOG_DIR)
print(f"Loading {len(resolved_files)} file(s) from LOG_DIR={LOG_DIR!r}:")
for p in resolved_files:
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print()

df, pec_config = load_pec(log_filenames, log_dir=LOG_DIR)
df = window_df(df, T_FROM, T_TO)
if T_FROM or T_TO:
    print(f"Window {T_FROM} .. {T_TO}: {len(df)} PECLOG rows")
if df.empty:
    raise ValueError(f"No PECLOG rows inside the window T_FROM={T_FROM!r} T_TO={T_TO!r}")

# Fallback if this log predates PECCONFIG (older sessions) — infer what we can
if pec_config is None:
    ra_model_cols = [c for c in df.columns if re.fullmatch(r'ra_model_\d+', c)]
    n_model_terms = len(ra_model_cols)
    if n_model_terms == 0:
        print("WARNING: no PECCONFIG line and no ra_model/dec_model columns found — "
              "cannot determine mode reliably. Defaulting to RLS/H=0, verify manually.")
        pec_config = dict(mode='rls', H=0, T=34*60, tau=21*60, min_dt=0.05)
    else:
        pec_config = dict(mode='rls', H=max(n_model_terms - 1, 0), T=34*60, tau=21*60, min_dt=0.05)
    print("No PECCONFIG line found — inferred:", pec_config)
else:
    print("PEC config from log:", pec_config)

print(f"Duration: {df['t_sec'].iloc[-1]/3600:.2f} hours")
print(f"N samples: {len(df)}")
print(f"Az range: {df['az'].min():.3f} to {df['az'].max():.3f}")
print()
df.columns


# Optional Save to CSV

In [ ]:
# df.to_csv('data.csv', index=False)

# Mount Orientation (Az, Alt, Roll) vs Time

These three plots show how Azimuth, Altitude and Roll Angle are changing over the session. 
Ideally they should trace smooth curves that are following the sidereal motion. If there are large steps then consider
dividing the log file into separate sessions as the mount has likely moved to a different target.

In [ ]:
df.columns

In [ ]:
# ── plot Az, Alt, Roll ───────────────────────────────────────────────────────────
xdata = df['t_sec'] / 60
xdata = df['timestamp'] 

def wrap_to_median(y, ref=None):
    """Re-express a circular (0-360) quantity as a signed deviation from `ref` (default: the
    data's own median), wrapped into (-180, 180] around it, then ref added back -- e.g. if
    the bulk of the samples sit near 0 but a few sit at 359, those become -1 instead of the
    0's staying put and the axis stretching out to fit 359. Using the plain numeric median
    (not a circular mean via atan2) is deliberate: atan2 always returns its result on the
    (-180, 180] branch, so a cluster that's genuinely near (say) 270 or 340 would come back
    as -90 or -20 and get added back at the *wrong* branch, pulling the whole recentered
    trace down near there instead of leaving it near 270/340. The median has no such branch
    to pick wrong -- it's just whichever raw sample sits in the middle once sorted, so as long
    as a wrap crossing is a minority of the samples (the normal case), it lands inside the
    real majority cluster on its own, wherever on the circle that happens to be. This also
    makes the trace continuous through a normal wrap -- the discontinuity was only ever in
    the 0-360 coordinate, not the mount's actual motion."""
    if ref is None:
        ref = y.median()
    return (y - ref + 180) % 360 - 180 + ref

def break_wraps(y, threshold=180):
    """NaN out a sample where it jumps by more than `threshold` degrees from the previous one
    so a line plot doesn't draw a spurious near-vertical segment straight across it. Run
    *after* wrap_to_median() -- that already makes an ordinary 0/360 wrap continuous, so
    anything still jumping this much here is a real, separate discontinuity (e.g. a slew to a
    new target -- see the markdown note above) worth flagging as a gap rather than a smooth
    line."""
    y = y.copy()
    y[y.diff().abs() > threshold] = np.nan
    return y

az_ref = df['az'].median()
print(f"Azimuth wrapped around its median ({az_ref:.2f}°)")

ydata = [
    #row, dataset,                                          name,                color
    (1,   break_wraps(wrap_to_median(df['az'], az_ref)),    'Azimuth (°, wrapped)', 'royalblue'),
    (2,   df['alt'],                                        'Altitude (°)',        'orange'),
    (3,   df['roll'],                                       'Roll (°)',            'mediumseagreen'),
]
rows = len(set([row for row, y,name,color in ydata]))
fig = make_subplots(rows=rows, cols=1, shared_xaxes=True,
    subplot_titles=[name for row, y,name,color in ydata],
    vertical_spacing=0.12)

for (row, y, name, color) in ydata:
    fig.add_trace(go.Scatter(x=xdata, y=y,
        mode='lines+markers', line=dict(color=color, width=0.8),
        name=name), row=row, col=1)
    fig.update_yaxes(title_text=name, row=row, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=3, col=1)
fig.update_layout(height=800, width=1200, template="plotly_dark",
        title='Range of Mount Orientation over session')
fig.show()

# PEC Outcome Overview

The real test of whether PEC is working: is `resid` (this sync's raw guide error, in arcmin --
`resid_1`=RA, `resid_2`=Dec after the standard list-flattening) trending toward zero and getting
less noisy over the session? This is the ground-truth guiding outcome PEC is trying to improve,
distinct from `total_accum` (which accumulates continuously by design and never shrinks -- see the
section below) and from `rmse`/`r2` (which describe the *model's own fit quality*, not the
resulting guiding error). Per control.py's `_pec_log()`: "should shrink as PEC improves."

Note `n` (and so `resid`) resets whenever the PEC model resets (goto/rotate/jog/stop-tracking/config
change -- see the reset-aware analysis further below), so a session spanning multiple resets will
show the residual jump back up at each reset before resuming its downward trend within that segment.

See also the Base vs Corrected Frame Shift diagnostic below, which cross-checks this from KFLOG telemetry when available.

In [ ]:
# ── PEC Outcome Overview: are guide residuals shrinking over the session? ──────────────────
# Each axis uses its own rows: a sync-guide row carries both RA and Dec, a pulse-guide row only one.
def axis_resid(col):
    r = df.dropna(subset=[col]).sort_values('t_sec').reset_index(drop=True)
    r['arcsec'] = r[col] * 60
    roll_n = max(5, len(r) // 20)   # ~20 points of smoothing across the session, floor of 5 cycles
    r['rms_roll'] = r['arcsec'].rolling(roll_n, min_periods=1).apply(lambda x: np.sqrt(np.mean(x**2)))
    return r, roll_n

ra_resid,  RA_ROLL_N  = axis_resid('resid_1')
dec_resid, DEC_ROLL_N = axis_resid('resid_2')

def trend_summary(t, y, label):
    """OLS slope of y (arcsec) vs t (sec) -- negative means shrinking/improving -- plus a
    first-fifth vs last-fifth RMS comparison, which is more robust to a single noisy cycle
    than the endpoints alone."""
    if len(t) < 4:
        print(f"{label}: not enough guide corrections ({len(t)}) for a trend")
        return
    slope, intercept = np.polyfit(t, y, 1)
    yhat = np.polyval([slope, intercept], t)
    r2 = r2_score(y, yhat)
    n = len(y)
    k = max(1, n // 5)
    rms_first = np.sqrt(np.mean(y[:k]**2))
    rms_last  = np.sqrt(np.mean(y[-k:]**2))
    pct_change = 100 * (rms_last - rms_first) / rms_first if rms_first != 0 else np.nan
    direction = "decreasing -- PEC improving" if slope < 0 else "increasing -- PEC not converging / worsening"
    print(f"=== {label} guide residual outcome (N={n} corrections) ===")
    print(f"Linear trend:         {slope*3600:+8.3f} arcsec/hr   R²={r2:.3f} ({r2_quality(r2)} fit) -- {direction}")
    print(f"First {k} RMS:          {rms_first:8.3f} arcsec")
    print(f"Last  {k} RMS:          {rms_last:8.3f} arcsec")
    print(f"Change:                {pct_change:+7.1f}%  ({'improvement' if pct_change < 0 else 'regression'})")
    print()

trend_summary(ra_resid.t_sec.values,  ra_resid.arcsec.values,  "RA")
trend_summary(dec_resid.t_sec.values, dec_resid.arcsec.values, "Dec")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["RA guide residual (arcsec) -- should trend toward 0 as PEC learns",
                     "Dec guide residual (arcsec) -- should trend toward 0 as PEC learns"],
    vertical_spacing=0.1)

fig.add_trace(go.Scatter(x=ra_resid.t_sec/60, y=ra_resid.arcsec, mode='markers', name='RA resid (raw)',
    marker=dict(size=4, color='royalblue', opacity=0.4)), row=1, col=1)
fig.add_trace(go.Scatter(x=ra_resid.t_sec/60, y=ra_resid.rms_roll, mode='lines', name=f'RA rolling RMS (n={RA_ROLL_N})',
    line=dict(color='orange', width=2)), row=1, col=1)

fig.add_trace(go.Scatter(x=dec_resid.t_sec/60, y=dec_resid.arcsec, mode='markers', name='Dec resid (raw)',
    marker=dict(size=4, color='mediumseagreen', opacity=0.4)), row=2, col=1)
fig.add_trace(go.Scatter(x=dec_resid.t_sec/60, y=dec_resid.rms_roll, mode='lines', name=f'Dec rolling RMS (n={DEC_ROLL_N})',
    line=dict(color='orange', width=2)), row=2, col=1)

fig.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=1, col=1)
fig.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=2, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=2, col=1)
fig.update_yaxes(title_text='arcsec', row=1, col=1)
fig.update_yaxes(title_text='arcsec', row=2, col=1)
fig.update_layout(height=700, width=1200, template='plotly_dark',
    title='PEC Outcome: Sync-Guide Residual Trend (raw resid + rolling RMS)',
    hovermode='x unified', legend=dict(groupclick='toggleitem'))
fig.show()

# PEC vs Guider

Who did the correcting? Within a PEC run the driver's own bookkeeping balances exactly:
**mount drift (`total_accum`) = guide corrections + PEC applied**, per axis. Running totals of each,
from PECLOG alone (no guider log needed):

- **Pulse-guide corrections** -- PHD2's pulses (rows carrying one axis)
- **Sync-guide corrections** -- plate-solve syncs (rows carrying both axes)
- **PEC applied** -- `pec_accum`, summed only on that axis's own rows (it is logged on every row
  but folded into the model only when that axis is ingested)
- **Mount drift** -- the sum of the three

PEC is helping when it carries most of the drift (**PEC share of drift** near 100%) and the guider's
total stays near zero. A negative share means PEC pushed the opposite way to the drift.
If PEC applied and the guider's corrections grow in opposite directions while the drift stays
small, the two are fighting: PEC pushes and the guider pulls it back.


In [ ]:
# ── PEC vs Guider: running totals per axis, from PECLOG ───────────────────────────────────────
is_sync_row = df['resid_1'].notna() & df['resid_2'].notna()      # sync guiding ingests both axes at once
resets = df.loc[df['n'].diff() < 0, 'timestamp']                  # PEC model resets (goto, config change, ...)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=['RA running totals (arcsec)', 'Dec running totals (arcsec)'])
summary = []
for row, (ax, name) in enumerate(((1, 'RA'), (2, 'Dec')), start=1):
    own = df[df[f'resid_{ax}'].notna()].copy()
    if own.empty:
        print(f"{name}: no guide corrections in PECLOG")
        continue
    corr = own[f'resid_{ax}'] * 60
    sync = is_sync_row.loc[own.index]
    own['pulse_guide'] = corr.where(~sync, 0.0).cumsum()
    own['sync_guide']  = corr.where(sync, 0.0).cumsum()
    own['pec_applied'] = (own[f'pec_accum_{ax}'] * 60).cumsum()
    own['drift']       = own['pulse_guide'] + own['sync_guide'] + own['pec_applied']
    for col, label, color in (('pulse_guide', 'pulse-guide corrections', 'royalblue'),
                              ('sync_guide',  'sync-guide corrections',  'orange'),
                              ('pec_applied', 'PEC applied',             'magenta'),
                              ('drift',       'mount drift',             'white')):
        fig.add_trace(go.Scatter(x=own['timestamp'], y=own[col], name=f'{name} {label}', legendgroup=name,
                                 line=dict(color=color, width=3 if col == 'drift' else 1.5)), row=row, col=1)
    last = own.iloc[-1]
    guider = last['pulse_guide'] + last['sync_guide']
    pec = last['pec_applied']
    undone = (-guider / pec * 100) if pec != 0 and np.sign(guider) != np.sign(pec) else 0.0
    share = pec / last['drift'] * 100 if last['drift'] != 0 else np.nan
    summary.append({'axis': name, 'pulse-guide "': round(last['pulse_guide']), 'sync-guide "': round(last['sync_guide']),
                    'PEC applied "': round(pec), 'mount drift "': round(last['drift']),
                    'PEC share of drift %': round(share), 'PEC undone by guider %': round(undone)})

for t in resets:
    fig.add_vline(x=t, line=dict(color='gray', width=1, dash='dot'))
fig.update_yaxes(title_text='arcsec', row=1, col=1)
fig.update_yaxes(title_text='arcsec', row=2, col=1)
fig.update_layout(height=750, width=1400, template='plotly_dark', hovermode='x unified',
    title='PEC vs Guider -- running totals (dotted lines: PEC model resets)')
fig.show()

summary = pd.DataFrame(summary)
print(summary.to_string(index=False))
for _, r in summary.iterrows():
    if r['PEC share of drift %'] < 0:
        print(f"{r['axis']}: PEC applied {r['PEC applied \"']:+}\" against a mount drift of {r['mount drift \"']:+}\" -- "
              f"PEC pushed the opposite way to the drift (check the correction sign)")
    if r['PEC undone by guider %'] > 50:
        print(f"{r['axis']}: the guider undid {r['PEC undone by guider %']}% of what PEC applied -- PEC and the guider are fighting")


## Guider Outcome (optional -- needs the PHD2 guide log)

Purpose: did guiding get better, and did the guider have to work less, once PEC was on? Uses only the
PHD2 guide log, in time blocks (`BLOCK`), so PEC-off and PEC-on periods on the same target can be compared.

- **Top chart -- guiding RMS (arcsec):** RA and Dec bars from PHD2's raw guide-star error, white line the
  combined total. Lower is better.
- **Bottom chart -- guider effort (pulse ms per minute):** total RA and Dec pulse time in both directions
  (at 1x guide rate, 1000 ms = 15"). If PEC is carrying the mount's drift, these bars should shrink once
  PEC is on. Effort includes chasing seeing, so the net direction is in the PEC vs Guider chart instead.
- **Shading** marks blocks with PEC on; **orange dotted lines** are plate-solve syncs. Each sync moves the
  model and the guider pulls the star back, so the summary also reports RMS excluding the minutes after
  each sync (`SYNC_SETTLE`).

Set `PHD2_GUIDELOG = None` to skip.


In [ ]:
# ── Guider outcome from the PHD2 guide log (optional) ─────────────────────────────────────────
BLOCK       = '20min'                  # time block for the table/chart
SYNC_SETTLE = pd.Timedelta('3min')     # frames this long after a sync are excluded from the on/off comparison

if not PHD2_GUIDELOG:
    print("No PHD2 guide log (PHD2_GUIDELOG = None) -- guider outcome skipped.")
else:
    phd2_path = PHD2_GUIDELOG if os.path.isabs(PHD2_GUIDELOG) else os.path.join(LOG_DIR, PHD2_GUIDELOG)
    phd2_frames, phd2_events = load_phd2_guidelog(phd2_path)
    phd2_frames = window_df(phd2_frames, T_FROM, T_TO)
    phd2_events = window_df(phd2_events, T_FROM, T_TO)
    if phd2_frames.empty:
        print(f"No PHD2 guided frames inside the window T_FROM={T_FROM!r} T_TO={T_TO!r}")
    else:
        fr = phd2_frames.copy()
        active = df[df['pec_active'].astype(bool)] if 'pec_active' in df.columns else df
        on = pd.merge_asof(fr[['timestamp']].astype('datetime64[ns]'),
                           active[['timestamp']].astype('datetime64[ns]').assign(pec_on=True),
                           on='timestamp', direction='nearest', tolerance=pd.Timedelta('60s'))
        fr['pec_on'] = on['pec_on'].astype('boolean').fillna(False).astype(bool).values
        # syncs from the always-logged SYNC GUIDING lines, so syncs while PEC was off count too
        try:
            sync_times = window_df(load_sync_guiding_residuals(log_filenames, log_dir=LOG_DIR), T_FROM, T_TO)['timestamp']
        except ValueError:
            sync_times = pd.Series([], dtype='datetime64[ns]')
        fr['after_sync'] = False
        for t in sync_times:
            fr.loc[(fr['timestamp'] >= t) & (fr['timestamp'] < t + SYNC_SETTLE), 'after_sync'] = True

        rms = lambda s: float(np.sqrt(np.mean(np.square(s)))) if len(s) else np.nan
        fr['block'] = fr['timestamp'].dt.floor(BLOCK)
        g = fr.groupby('block')
        minutes = g['timestamp'].agg(lambda s: max((s.iloc[-1] - s.iloc[0]).total_seconds() / 60, 1e-9))
        lost = phd2_events[phd2_events['kind'] == 'star_lost'].groupby(
            phd2_events['timestamp'].dt.floor(BLOCK)).size() if not phd2_events.empty else pd.Series(dtype=int)
        blocks = pd.DataFrame({
            'PEC on %':        (g['pec_on'].mean() * 100).round(0),
            'RA rms "':        g['ra_arcsec'].agg(rms).round(2),
            'Dec rms "':       g['dec_arcsec'].agg(rms).round(2),
            'total rms "':     g.apply(lambda x: rms(np.hypot(x['ra_arcsec'], x['dec_arcsec'])), include_groups=False).round(2),
            'RA pulses ms/min':  (g['ra_ms'].agg(lambda s: s.abs().sum()) / minutes).round(0),
            'Dec pulses ms/min': (g['dec_ms'].agg(lambda s: s.abs().sum()) / minutes).round(0),
            'syncs':           sync_times.groupby(sync_times.dt.floor(BLOCK)).size(),
            'star lost':       lost,
        }).loc[minutes.index].fillna({'syncs': 0, 'star lost': 0}).astype({'syncs': int, 'star lost': int})
        print(blocks.to_string())
        print()

        for label, sel in (('PEC off', ~fr['pec_on']), ('PEC on', fr['pec_on'])):
            for excl, mask in (('all frames', sel), (f'excluding {SYNC_SETTLE.total_seconds()/60:.0f} min after syncs', sel & ~fr['after_sync'])):
                x = fr[mask]
                if len(x):
                    print(f"{label:7s} {excl:30s} RA {rms(x['ra_arcsec']):.2f}\"  Dec {rms(x['dec_arcsec']):.2f}\"  "
                          f"total {rms(np.hypot(x['ra_arcsec'], x['dec_arcsec'])):.2f}\"  ({len(x)} frames)")

        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
            subplot_titles=[f'Guiding RMS per {BLOCK} (arcsec)', 'Guider effort (pulse ms per minute)'])
        fig.add_trace(go.Bar(x=blocks.index, y=blocks['RA rms "'], name='RA rms', marker_color='royalblue'), row=1, col=1)
        fig.add_trace(go.Bar(x=blocks.index, y=blocks['Dec rms "'], name='Dec rms', marker_color='mediumseagreen'), row=1, col=1)
        fig.add_trace(go.Scatter(x=blocks.index, y=blocks['total rms "'], name='total rms', mode='lines+markers',
                                 line=dict(color='white')), row=1, col=1)
        fig.add_trace(go.Bar(x=blocks.index, y=blocks['RA pulses ms/min'], name='RA pulses', marker_color='royalblue',
                             showlegend=False), row=2, col=1)
        fig.add_trace(go.Bar(x=blocks.index, y=blocks['Dec pulses ms/min'], name='Dec pulses', marker_color='mediumseagreen',
                             showlegend=False), row=2, col=1)
        for t in sync_times:
            fig.add_vline(x=t, line=dict(color='orange', width=1, dash='dot'))
        on_blocks = blocks.index[blocks['PEC on %'] >= 50]
        for b in on_blocks:
            fig.add_vrect(x0=b, x1=b + pd.Timedelta(BLOCK), fillcolor='magenta', opacity=0.08, line_width=0)
        fig.update_layout(height=750, width=1400, template='plotly_dark', barmode='group', hovermode='x unified',
            title='Guider Outcome (shaded: PEC on, dotted: syncs)')
        fig.show()


## Base vs Corrected Frame Shift (optional -- requires KFLOG)

A second, independent cross-check on the same question, from KFLOG telemetry instead of
PECLOG: `θ_meas_raw` (Base Frame -- the raw, uncorrected motor encoder position) vs
`θ_ref_raw` (Corrected Frame -- the PID's target motor position, which already includes
PEC's feed-forward rate). If PEC is genuinely absorbing the periodic error, this gap should
be a smooth curve, without any many major corrections from the sync-guide events. Any 
sawtooth or sharp jumps in these plots indicates sync-guiding is correcting the mounts
PE or the drivers PE corrections.

KFLOG is gated by `Config.log_position`, independently of PECLOG's `Config.log_pec`, so it
isn't present in every capture -- this section is skipped with a message when it's missing.

In [ ]:
# ── Optional: Base Frame vs Corrected Frame shift per motor axis, from KFLOG ───────────────
kf_df, _pid_df_unused = load_kf_pid(log_filenames, log_dir=LOG_DIR)
kf_df = window_df(kf_df, T_FROM, T_TO)

_AXES = ["M1", "M2", "M3"]
_shift_cols = [f'\u03b8_meas_raw_{i}' for i in (1, 2, 3)] + [f'\u03b8_ref_raw_{i}' for i in (1, 2, 3)]

if not len(kf_df) or not all(c in kf_df.columns for c in _shift_cols):
    print("No KFLOG \u03b8_meas_raw/\u03b8_ref_raw data in this capture -- Base/Corrected frame "
          "shift diagnostic unavailable (needs Config.log_position=true; independent of "
          "Config.log_pec, so this is expected on a lot of PEC-only captures).")
else:
    kf = kf_df.dropna(subset=_shift_cols).sort_values('t_sec').reset_index(drop=True)

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
        subplot_titles=[f'{ax}: Base Frame (\u03b8_meas_raw) \u2212 Corrected Frame (\u03b8_ref_raw)  (arcsec)' for ax in _AXES],
        vertical_spacing=0.08)

    print(f"=== Base/Corrected frame shift summary (arcsec, N={len(kf)} KFLOG samples) ===")
    for i, ax in enumerate(_AXES, start=1):
        # wrap into (-180, 180] before converting to arcsec -- these are absolute motor
        # positions, so a raw difference would spuriously jump by ~360deg whenever one side
        # crosses the wrap point the other hasn't yet.
        shift_deg = kf[f'\u03b8_meas_raw_{i}'] - kf[f'\u03b8_ref_raw_{i}']
        shift_arcsec = ((shift_deg + 180) % 360 - 180) * ARCSEC

        fig.add_trace(go.Scatter(x=kf.timestamp, y=shift_arcsec, name=f'{ax} shift',
            mode='lines', line=dict(width=1)), row=i, col=1)

        n = len(shift_arcsec)
        k = max(1, n // 5)
        rms_first = np.sqrt(np.mean(shift_arcsec.values[:k]**2))
        rms_last  = np.sqrt(np.mean(shift_arcsec.values[-k:]**2))
        pct = 100 * (rms_last - rms_first) / rms_first if rms_first != 0 else np.nan
        print(f"{ax}: first {k} samples RMS={rms_first:8.2f}\"   last {k} samples RMS={rms_last:8.2f}\"   "
              f"change={pct:+6.1f}%  ({'improvement' if pct < 0 else 'regression'})")

    fig.update_xaxes(title_text='Time (minutes)', row=3, col=1)
    fig.update_layout(height=800, width=1200, template='plotly_dark',
        title='PEC Outcome: Base Frame vs Corrected Frame Shift per Motor Axis (KFLOG)',
        hovermode='x unified')
    fig.show()

# Analysis of PECLOG

In [ ]:
# ── plot rows 1–4 ───────────────────────────────────────────────────────────
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from control import PecAxis, PecInhibit, PecMode

# ── calc INTERVAL (diagnostic only — no longer drives lambda/alpha directly) ──
ra_times = df.dropna(subset=['resid_1'])['t_sec']
INTERVAL = ra_times.diff().dropna().median()

# ── instantiate from parsed session config ────────────────────────────────
T        = pec_config['T']
H        = pec_config['H']
TAU      = pec_config['tau']
MIN_DT   = pec_config['min_dt']
MODE     = PecMode(pec_config['mode']) if pec_config['mode'] in ('rls', 'ema') else PecMode.RLS
VAR      = 0.05
SSE      = 0.15
COLORS   = ['green', 'purple', 'orange', 'red', 'cyan', 'magenta']
DEG_S_TO_ARCMIN_HR = 3600 * 60   # deg/sec to arcmin/hr

ra  = PecAxis(T=T, n_harmonics=H, mode=MODE, tau=TAU, min_dt=MIN_DT)
dec = PecAxis(T=T, n_harmonics=H, mode=MODE, tau=TAU, min_dt=MIN_DT)
var_alpha = VAR
sse_alpha = SSE

ra.reset_seed(df.iloc[0].total_accum_1 / 60)
dec.reset_seed(df.iloc[0].total_accum_2 / 60)

t0 = 0.0            # local time origin — shifts forward on every detected PEC reset
prev_n = df.iloc[0].n

show_harmonics = (MODE == PecMode.RLS and H > 0)

records = []
reset_events = []   # for plotting/inspection — t_min of each detected reset
reset_indices = []  # for fixing gradient

for i, row in df.iterrows():
    t_abs     = row.t_sec
    ra_cumul  = row.total_accum_1  / 60
    dec_cumul = row.total_accum_2 / 60

    # ── detect a PEC reset from the guide counter rolling back ──
    if row.n < prev_n:
        ra.reset()
        dec.reset()
        ra.reset_seed(ra_cumul)
        dec.reset_seed(dec_cumul)
        t0 = t_abs                      # restart local clock — matches driver's _pec_t0 reset
        reset_events.append(t_abs / 60)
        reset_indices.append(i)
    prev_n = row.n

    t = t_abs - t0                      # local time, relative to most recent reset

    ra.ingest_accum(ra_cumul, t, var_alpha, sse_alpha)
    dec.ingest_accum(dec_cumul, t, var_alpha, sse_alpha)

    records.append(dict(
        t_min    = t_abs / 60,          # keep x-axis on session-absolute time for plotting continuity
        timestamp = row.timestamp,
        n        = row.n,
        ra_rate  = ra.predicted_rate(t)  * DEG_S_TO_ARCMIN_HR,
        dec_rate = dec.predicted_rate(t) * DEG_S_TO_ARCMIN_HR,
        ra_drift = ra.dc_rate()   * DEG_S_TO_ARCMIN_HR,
        dec_drift= dec.dc_rate()  * DEG_S_TO_ARCMIN_HR,
        **{f'ra_h{h}':  ra.harmonic_rate(h)  * DEG_S_TO_ARCMIN_HR for h in range(1, H+1)},
        **{f'dec_h{h}': dec.harmonic_rate(h) * DEG_S_TO_ARCMIN_HR for h in range(1, H+1)},
        ra_r2    = ra.r2,
        dec_r2   = dec.r2,
        ra_rmse  = ra.rmse_arcmin(),
        dec_rmse = dec.rmse_arcmin(),
        ra_pred  = ra.predicted_accum(t)  * 60,
        dec_pred = dec.predicted_accum(t) * 60,
        ra_cumul = ra_cumul  * 60,
        dec_cumul= dec_cumul * 60,
        ra_resid_fit  = (ra_cumul*60)  - ra.predicted_accum(t)*60,
        dec_resid_fit = (dec_cumul*60) - dec.predicted_accum(t)*60,
        ra_lambda  = ra.lam,
        dec_lambda = dec.lam,
    ))

res = pd.DataFrame(records)
t   = res.t_min
t   = res.timestamp

# ── empirical (actual) drift rate ──
res['ra_actual_rate']  = np.full(len(res), np.nan)
res['dec_actual_rate'] = np.full(len(res), np.nan)

reset_indices = sorted(set([0] + reset_indices + [len(res)]))
for start, end in zip(reset_indices[:-1], reset_indices[1:]):
    if end - start < 2:
        continue   # need at least 2 points to take a gradient
    seg = res.iloc[start:end]
    seg_t = seg.t_min.values * 60           # ra_cumul in arcmin; seg_t in seconds; ra_actual_rate in arcmin/hr
    res.loc[seg.index, 'ra_actual_rate']  = np.gradient(seg.ra_cumul.values,  seg_t) * 3600
    res.loc[seg.index, 'dec_actual_rate'] = np.gradient(seg.dec_cumul.values, seg_t) * 3600

mode_label = MODE.value.upper()
title_text = (f'PEC Model (mode={mode_label}   Hₙ={H}   Period₁={T/60:.0f} min   '
              f'dt_med={INTERVAL:.1f}s   τ={TAU:.0f}s   sse_α={SSE}   var_α={VAR})')

# ── plot rows 1–4 ───────────────────────────────────────────────────────────
n_rows_top = 4
row_titles_top = [
    'PEC guide counter (resets to 1 after goto)',            'PEC guide counter (resets to 1 after goto)',
    'RA total_accum and fit (arcmin)',                       'Dec total_accum and fit (arcmin)',
    'RA predicted vs actual drift rate (arcmin/hr)',         'Dec predicted vs actual drift rate (arcmin/hr)',
    'RA predicted rate breakdown: DC + harmonics (arcmin/hr)','Dec predicted rate breakdown: DC + harmonics (arcmin/hr)',
]

fig_top = make_subplots(
    rows=n_rows_top, cols=2,
    shared_xaxes=True,
    subplot_titles=tuple(row_titles_top),
    vertical_spacing=0.08,
)

def add(fig, row, col, traces):
    for tr in traces:
        fig.add_trace(tr, row=row, col=col)

# row 1 — counter
add(fig_top, 1, 1, [go.Scatter(x=t, y=res.n, name='n', mode='lines+markers', line=dict(color='royalblue'))])
add(fig_top, 1, 2, [go.Scatter(x=t, y=res.n, name='n', mode='lines+markers', line=dict(color='royalblue'))])

# row 2 — cumul fit
add(fig_top, 2, 1, [
    go.Scatter(x=t, y=res.ra_cumul, name='RA actual',    mode='lines+markers', opacity=0.6, line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.ra_pred,  name='RA predicted', line=dict(color='red', width=2)),
])
add(fig_top, 2, 2, [
    go.Scatter(x=t, y=res.dec_cumul, name='Dec actual',    mode='lines+markers', opacity=0.6, line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.dec_pred,  name='Dec predicted', line=dict(color='red', width=2)),
])

# row 3 — predicted vs actual total drift rate
add(fig_top, 3, 1, [
    go.Scatter(x=t, y=res.ra_rate,        name='RA Predicted Rate', line=dict(color='orange')),
    go.Scatter(x=t, y=res.ra_actual_rate, name='RA Actual Rate',    mode='lines', opacity=0.6, line=dict(color='royalblue')),
])
add(fig_top, 3, 2, [
    go.Scatter(x=t, y=res.dec_rate,        name='Dec Predicted Rate', line=dict(color='orange')),
    go.Scatter(x=t, y=res.dec_actual_rate, name='Dec Actual Rate',    mode='lines', opacity=0.6, line=dict(color='royalblue')),
])

# row 4 — breakdown of predicted rate: DC + harmonics
add(fig_top, 4, 1, [
    go.Scatter(x=t, y=res.ra_drift, name='RA DC Component', line=dict(color='#636EFA', dash='dash')),
] + ([
    go.Scatter(x=t, y=res[f'ra_h{h}'], name=f'RA H{h} Component', line=dict(color=COLORS[(h-1) % len(COLORS)], dash='dash'))
    for h in range(1, H+1)
] if show_harmonics else []))
add(fig_top, 4, 2, [
    go.Scatter(x=t, y=res.dec_drift, name='Dec DC Component', line=dict(color='#636EFA', dash='dash')),
] + ([
    go.Scatter(x=t, y=res[f'dec_h{h}'], name=f'Dec H{h} Component', line=dict(color=COLORS[(h-1) % len(COLORS)], dash='dash'))
    for h in range(1, H+1)
] if show_harmonics else []))

fig_top.update_xaxes(matches="x")   # link every subplot's x-axis to the first, across both columns
fig_top.update_layout(
    height=300 * n_rows_top,
    title=dict(text=title_text, x=0.5),
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig_top.update_xaxes(title_text='Time (minutes)', row=n_rows_top)

fig_top.show()


In [ ]:
# ── plot rows 5–6 ───────────────────────────────────────────────────────────
row5_min = 0
n_rows_bottom = 2
row_titles_bottom = [
    'RA fit R² quality',                      'Dec fit R² quality',
    'RA fit residuals and rmse (arcmin)',     'Dec fit residuals and rmse (arcmin)',
]

fig_bottom = make_subplots(
    rows=n_rows_bottom, cols=2,
    shared_xaxes=True,
    subplot_titles=tuple(row_titles_bottom),
    vertical_spacing=0.12,
)

def add(fig, row, col, traces):
    for tr in traces:
        fig.add_trace(tr, row=row, col=col)

# row 1 (of this fig) — R²
add(fig_bottom, 1, 1, [go.Scatter(x=t, y=res.ra_r2,  name='RA R²',  line=dict(color='royalblue'))])
add(fig_bottom, 1, 2, [go.Scatter(x=t, y=res.dec_r2, name='Dec R²', line=dict(color='royalblue'))])

# row 2 (of this fig) — residuals
add(fig_bottom, 2, 1, [
    go.Scatter(x=t, y=res.ra_resid_fit, name='RA Residuals', line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.ra_rmse,      name='RA rmse',      line=dict(color='red', dash='dash')),
])
add(fig_bottom, 2, 2, [
    go.Scatter(x=t, y=res.dec_resid_fit, name='Dec Residuals', line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.dec_rmse,      name='Dec rmse',      line=dict(color='red', dash='dash')),
])

fig_bottom.add_hrect(y0=row5_min, y1=0.5, fillcolor='rgba(255,0,0,0.15)', line_width=0, row=1, col=1)
fig_bottom.add_hrect(y0=row5_min, y1=0.5, fillcolor='rgba(255,0,0,0.15)', line_width=0, row=1, col=2)
fig_bottom.update_yaxes(range=[row5_min, 1.0], row=1, col=1)
fig_bottom.update_yaxes(range=[row5_min, 1.0], row=1, col=2)
fig_bottom.update_xaxes(matches="x")   # link every subplot's x-axis to the first, across both columns

fig_bottom.update_layout(
    height=300 * n_rows_bottom,
    title=dict(text=title_text, x=0.5),
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig_bottom.update_xaxes(title_text='Time (minutes)', row=n_rows_bottom)
fig_bottom.show()


# Cumulative Total Drift (total_accum) vs Time

In [ ]:
# ── plot accumulated corrections single chart ───────────────────────────────────────────────────────────
fig = go.Figure()
fig.update_layout(
    height=600 , width=1800,
    title=dict(text="RA Axis - Cumulative Total Drift (total_accum) vs Time", x=0.5, font_size=32),
    xaxis=dict(title="Time (minutes)", title_font=dict(size=20)),
    yaxis=dict(title="Total Accumulated Drift (arcmin)", title_font=dict(size=20)),
    legend_font_size=20,
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig.add_trace(go.Scatter(x=t, y=res.ra_cumul, name='RA actual',    mode='lines+markers', opacity=0.6, line=dict(color='white')))
fig.add_trace(go.Scatter(x=t, y=res.ra_pred,  name='RA predicted', line=dict(color='green', width=2)))
fig.show()

fig = go.Figure()
fig.update_layout(
    height=600 , width=1800,
    title=dict(text="Dec Axis - Cumulative Total Drift (total_accum) vs Time", x=0.5, font_size=32),
    xaxis=dict(title="Time (minutes)", title_font=dict(size=20)),
    yaxis=dict(title="Total Accumulated Drift (arcmin)", title_font=dict(size=20)),
    legend_font_size=20,
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig.add_trace(go.Scatter(x=t, y=res.dec_cumul, name='Dec actual',    mode='lines+markers', opacity=0.6, line=dict(color='white')))
fig.add_trace(go.Scatter(x=t, y=res.dec_pred,  name='Dec predicted', line=dict(color='green', width=2)))
fig.show()

# Right Ascension Drift

In [ ]:
t = df['t_sec'].values
dec = df['total_accum_1'].values
poly_coeffs, residuals, rank, sv, rcond = np.polyfit(t, dec, 1, full=True)
slope, intercept = poly_coeffs
dec_fit = np.polyval(poly_coeffs, t)
n = len(t)
rss = residuals[0] if len(residuals) > 0 else np.sum((dec - dec_fit)**2)
tss = np.sum((dec - np.mean(dec))**2)
r2 = 1 - rss / tss if tss != 0 else float('nan')
rmse = np.sqrt(rss / n)
print(f"=== Right Ascension - Drift Fit Summary (N={n}) ===")
print(f"Slope (drift rate): {slope*3600:+8.3f} arcmin/hr")
print(f"Intercept:          {intercept:+8.3f} arcmin")
print(f"RMSE (noise):       {rmse:8.3f} arcmin")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")

# Declination Drift

In [ ]:
t = df['t_sec'].values
dec = df['total_accum_2'].values
poly_coeffs, residuals, rank, sv, rcond = np.polyfit(t, dec, 1, full=True)
slope, intercept = poly_coeffs
dec_fit = np.polyval(poly_coeffs, t)
n = len(t)
rss = residuals[0] if len(residuals) > 0 else np.sum((dec - dec_fit)**2)
tss = np.sum((dec - np.mean(dec))**2)
r2 = 1 - rss / tss if tss != 0 else float('nan')
rmse = np.sqrt(rss / n)
print(f"=== Declination - Drift Fit Summary (N={n}) ===")
print(f"Slope (drift rate): {slope*3600:+8.3f} arcmin/hr")
print(f"Intercept:          {intercept:+8.3f} arcmin")
print(f"RMSE (noise):       {rmse:8.3f} arcmin")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")


# Right Ascension Period

In [ ]:
from scipy.signal import periodogram
import numpy as np

# RA: the periodic PEC signal
ra = df['total_accum_1'].values

# Remove linear drift (same idea as DEC fit)
ra_detrended = ra - np.polyval(np.polyfit(t, ra, 1), t)

# Sampling frequency
fs = 1 / np.median(np.diff(t))

# Periodogram
freqs, power = periodogram(ra_detrended, fs=fs)

# Ignore zero frequency and periods > 100 min 
max_period_sec = 120 * 60
min_freq = 1 / max_period_sec
valid = (freqs >= min_freq)
freqs = freqs[valid]
power = power[valid]

# Peak detection
peak_idx = np.argmax(power)
peak_freq = freqs[peak_idx]
peak_power = power[peak_idx]

worm_period_sec = 1 / peak_freq

# Noise floor estimate (median is robust)
mask = np.ones_like(power, dtype=bool)
window = 3  # exclude ±3 bins around peak
mask[max(0, peak_idx-window):peak_idx+window+1] = False
noise_floor = np.median(power[mask])

# Signal-to-noise ratio
snr = peak_power / noise_floor if noise_floor > 0 else np.inf
snr_db = 10 * np.log10(peak_power / noise_floor)

# Estimate amplitude (rough, from detrended signal)
amp = (np.max(ra_detrended) - np.min(ra_detrended)) / 2

# Output
n = len(ra)

print(f"=== Right Ascension - PEC Analysis (N={n}) ===")
print(f"Worm period:        {worm_period_sec/60:8.3f} min")
print(f"Amplitude:          {amp*60:8.3f} arcmin")
print(f"Peak power:         {10*np.log10(peak_power):8.3f} dB (relative)")
print(f"Noise floor:        {10*np.log10(noise_floor):8.3f} dB (relative)")
print(f"SNR (periodicity):  {snr_db:8.2f} dB {pec_quality_db(snr_db)} signal")

In [ ]:
periods_min = 1 / freqs / 60
log_power = np.log10(power)
max_y = np.max(log_power)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=periods_min,
    y=log_power,
    mode='lines',
    name='log10(Power)'
))

fig.add_vline(x=worm_period_sec/60, line_dash="dash", line_color="red")

fig.update_layout(
    title="RA Periodogram (Log Power)",
    xaxis_title="Period (minutes)",
    yaxis_title="log10(Power)", 
    height=600, width=1200, template="plotly_dark",
)
max_y
fig.update_yaxes(range=[ -4, max_y])

fig.show()

# Declination Period

In [ ]:
from scipy.signal import periodogram
import numpy as np

# Dec: the periodic PEC signal
dec = df['total_accum_2'].values

# Remove linear drift (same idea as DEC fit)
dec_detrended = dec - np.polyval(np.polyfit(t, dec, 1), t)

# Sampling frequency
fs = 1 / np.median(np.diff(t))

# Periodogram
freqs, power = periodogram(dec_detrended, fs=fs)

# Ignore zero frequency and periods > 100 min 
max_period_sec = 120 * 60
min_freq = 1 / max_period_sec
valid = (freqs >= min_freq)
freqs = freqs[valid]
power = power[valid]

# Peak detection
peak_idx = np.argmax(power)
peak_freq = freqs[peak_idx]
peak_power = power[peak_idx]

worm_period_sec = 1 / peak_freq

# Noise floor estimate (median is robust)
mask = np.ones_like(power, dtype=bool)
window = 3  # exclude ±3 bins around peak
mask[max(0, peak_idx-window):peak_idx+window+1] = False
noise_floor = np.median(power[mask])

# Signal-to-noise ratio
snr = peak_power / noise_floor if noise_floor > 0 else np.inf
snr_db = 10 * np.log10(peak_power / noise_floor)

# Estimate amplitude (rough, from detrended signal)
amp = (np.max(ra_detrended) - np.min(ra_detrended)) / 2

# Output
n = len(dec)

print(f"=== Declination - PEC Analysis (N={n}) ===")
print(f"Worm period:        {worm_period_sec/60:8.3f} min")
print(f"Amplitude:          {amp*60:8.3f} arcmin")
print(f"Peak power:         {10*np.log10(peak_power):8.3f} dB (relative)")
print(f"Noise floor:        {10*np.log10(noise_floor):8.3f} dB (relative)")
print(f"SNR (periodicity):  {snr_db:8.2f} dB {pec_quality_db(snr_db)} signal")


In [ ]:
periods_min = 1 / freqs / 60
log_power = np.log10(power)
max_y = np.max(log_power)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=periods_min,
    y=log_power,
    mode='lines',
    name='log10(Power)'
))

fig.add_vline(x=worm_period_sec/60, line_dash="dash", line_color="red")

fig.update_layout(
    title="Dec Periodogram (Log Power)",
    xaxis_title="Period (minutes)",
    yaxis_title="log10(Power)", 
    height=600, width=1200, template="plotly_dark",
)
max_y
fig.update_yaxes(range=[ -4, max_y])

fig.show()

# Notes